In [20]:
import pandas as pd
from global_macro_data import gmd

In [28]:
class GMDClient:
    def __init__(self):
        pass
    def get_gmd(self):
        #return gmd()
        return pd.read_csv(r"C:\Diversification\data\GMD.csv") #gmd(version="2026_03")
    def validate_type(self, gmd_df):
        if not isinstance(gmd_df, pd.DataFrame):
            raise TypeError(f"GMDClient expected pandas.DataFrame got {type(gmd_df)}")
        if gmd_df.empty:
            raise ValueError(f"GMD Dataframe ist empty")
        return gmd_df
    def run(self):
            df = self.get_gmd()
            return self.validate_type(df)
    
#GMDClient().run()

In [29]:
class GMDTransformer:
    def __init__(self):
        pass
    def clean_id(self, df):
        df = df.drop(columns=['id'])
        return df
    def objects_to_int(self, df):
        income_map = {"Low income": 1, "Lower middle income": 2, "Upper middle income": 3, "High income": 4,}
        df["income_group_code"] = df["income_group"].map(income_map).astype("Int8")
        df = df.drop(columns=['income_group'])
        return df
    def to_long(self, df):
        id_vars = ["countryname", "ISO3", "year"]
        df_long = df.melt(id_vars=id_vars, var_name="Item_Description", value_name="Value")
        return df_long
    def run(self, df):
        df = self.clean_id(df)
        df = self.objects_to_int(df)
        df = self.to_long(df)
        return df

In [30]:
class GMDIngestor:
    def __init__(self):
        self.client = GMDClient()
        self.transformer = GMDTransformer()
    def run(self):
        df = self.client.run()
        df = self.transformer.run(df)
        return df

In [44]:
gmd_df = GMDIngestor().run()

In [45]:
gmd_df

,countryname,ISO3,year,Item_Description,Value
0,Aruba,ABW,1960.0,nGDP,<NA>
1,Aruba,ABW,1961.0,nGDP,<NA>
2,Aruba,ABW,1962.0,nGDP,<NA>
3,Aruba,ABW,1963.0,nGDP,<NA>
4,Aruba,ABW,1964.0,nGDP,<NA>
...,...,...,...,...,...
4549115,Zimbabwe,ZWE,2027.0,income_group_code,2.0
4549116,Zimbabwe,ZWE,2028.0,income_group_code,2.0
4549117,Zimbabwe,ZWE,2029.0,income_group_code,2.0
4549118,Zimbabwe,ZWE,2030.0,income_group_code,2.0


In [2]:
from pathlib import Path
import duckdb
import pandas as pd


class DataHub:
    def __init__(self, db_name="bronze.db"):
        # Pfad-Management (funktioniert in Scripts & Notebooks)
        base_path = Path.cwd()
        db_path = base_path.parent.parent / "data" / db_name
        db_path.parent.mkdir(parents=True, exist_ok=True)

        # Verbindung herstellen
        self.con = duckdb.connect(str(db_path))
        self._initialize_tables()
        print(f"DuckDB verbunden: {db_path}")

    def _initialize_tables(self):
        """Erstellt die Tabellenstruktur, falls sie noch nicht existiert."""

        # Yahoo / Financials
        self.con.execute("""
            CREATE TABLE IF NOT EXISTS bronze_financials (
                ticker VARCHAR,
                date DATE,
                affiliation VARCHAR,
                item_description VARCHAR,
                value DOUBLE,
                ingested_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP
            );
        """)
        self.con.execute("""
            CREATE INDEX IF NOT EXISTS idx_ticker_date
            ON bronze_financials (ticker, date);
        """)

        # Wikidata
        self.con.execute("""
            CREATE TABLE IF NOT EXISTS bronze_wikidata (
                company_qid VARCHAR,
                item_description VARCHAR,
                value VARCHAR,
                ingested_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP
            );
        """)
        self.con.execute("""
            CREATE INDEX IF NOT EXISTS idx_wikidata_qid
            ON bronze_wikidata (company_qid);
        """)
        self.con.execute("""
            CREATE INDEX IF NOT EXISTS idx_wikidata_qid_item
            ON bronze_wikidata (company_qid, item_description);
        """)

        # GMD
        self.con.execute("""
            CREATE TABLE IF NOT EXISTS bronze_gmd (
                countryname VARCHAR,
                iso3 VARCHAR,
                year INTEGER,
                item_description VARCHAR,
                value DOUBLE,
                ingested_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP
            );
        """)
        self.con.execute("""
            CREATE INDEX IF NOT EXISTS idx_gmd_iso3_year
            ON bronze_gmd (iso3, year);
        """)
        self.con.execute("""
            CREATE INDEX IF NOT EXISTS idx_gmd_item
            ON bronze_gmd (item_description);
        """)

    def insert_financials(self, df: pd.DataFrame):
        """Speichert Yahoo-Financials in die Datenbank."""
        if df is None or df.empty:
            return

        df = df.copy()

        try:
            self.con.execute("""
                INSERT INTO bronze_financials (
                    ticker, date, affiliation, item_description, value
                )
                SELECT
                    ticker, date, affiliation, item_description, value
                FROM df
            """)
        except Exception as e:
            print(f"Fehler beim Insert in bronze_financials: {e}")

    def insert_wikidata(self, df: pd.DataFrame):
        """
        Speichert Wikidata-DataFrame in die Datenbank.
        Erwartete Spalten: company / Company_QID, Item_Description, Value
        """
        if df is None or df.empty:
            return

        df = df.copy()

        # Flexible Behandlung der QID-Spalte
        if "Company_QID" not in df.columns:
            if "company" in df.columns:
                df["Company_QID"] = df["company"].astype(str)
            else:
                raise ValueError(
                    f"Wikidata DF braucht 'Company_QID' oder 'company'. Vorhanden: {list(df.columns)}"
                )

        required = {"Company_QID", "Item_Description", "Value"}
        if not required.issubset(df.columns):
            missing = required - set(df.columns)
            raise ValueError(
                f"Wikidata DF fehlt Spalten: {missing}. Vorhanden: {list(df.columns)}"
            )

        df["Company_QID"] = df["Company_QID"].astype(str)
        df["Item_Description"] = df["Item_Description"].astype(str)
        df["Value"] = df["Value"].astype(str)

        try:
            self.con.execute("""
                INSERT INTO bronze_wikidata (
                    company_qid, item_description, value
                )
                SELECT
                    Company_QID AS company_qid,
                    Item_Description AS item_description,
                    Value AS value
                FROM df
            """)
        except Exception as e:
            print(f"Fehler beim Insert in bronze_wikidata: {e}")

    def insert_gmd(self, df: pd.DataFrame):
        """
        Speichert GMD-Long-DataFrame in die Datenbank.

        Erwartete Spalten:
        countryname | ISO3 | year | Item_Description | Value
        """
        if df is None or df.empty:
            return

        df = df.copy()

        required = {"countryname", "ISO3", "year", "Item_Description", "Value"}
        if not required.issubset(df.columns):
            missing = required - set(df.columns)
            raise ValueError(
                f"GMD DF fehlt Spalten: {missing}. Vorhanden: {list(df.columns)}"
            )

        # Typen normalisieren
        df["countryname"] = df["countryname"].astype(str)
        df["ISO3"] = df["ISO3"].astype(str)
        df["year"] = pd.to_numeric(df["year"], errors="coerce").astype("Int64")
        df["Item_Description"] = df["Item_Description"].astype(str)
        df["Value"] = pd.to_numeric(df["Value"], errors="coerce")

        # Optional: Zeilen ohne year verwerfen
        df = df[df["year"].notna()].copy()
        df["year"] = df["year"].astype(int)

        try:
            self.con.execute("""
                INSERT INTO bronze_gmd (
                    countryname, iso3, year, item_description, value
                )
                SELECT
                    countryname,
                    ISO3 AS iso3,
                    year,
                    Item_Description AS item_description,
                    Value AS value
                FROM df
            """)
        except Exception as e:
            print(f"Fehler beim Insert in bronze_gmd: {e}")

    def preview_data(self, table="bronze_gmd", limit=5000000):
        """Holt Einträge als DataFrame zur Kontrolle."""
        return self.con.execute(
            f"SELECT * FROM {table} LIMIT {int(limit)}"
        ).df()

    def get_summary_stats(self):
        """Gibt eine kleine Statistik über den Füllstand der DB aus."""
        return self.con.execute("""
            SELECT
                (SELECT COUNT(DISTINCT ticker) FROM bronze_financials) AS count_tickers,
                (SELECT COUNT(*) FROM bronze_financials) AS total_financial_rows,
                (SELECT COUNT(DISTINCT company_qid) FROM bronze_wikidata) AS count_company_qids,
                (SELECT COUNT(*) FROM bronze_wikidata) AS total_wikidata_rows,
                (SELECT COUNT(DISTINCT iso3) FROM bronze_gmd) AS count_gmd_countries,
                (SELECT COUNT(*) FROM bronze_gmd) AS total_gmd_rows
        """).df()

    def close(self):
        """Schließt die Verbindung sauber."""
        self.con.close()

    def clear_database(self):
        """Löscht alle Daten und Tabellen aus der DuckDB-Datenbank."""
        try:
            tables = self.con.execute("SHOW TABLES").fetchall()

            if not tables:
                print("Datenbank ist bereits leer.")
                return

            print(f"Lösche {len(tables)} Tabellen...")

            for (table_name,) in tables:
                self.con.execute(f"DROP TABLE IF EXISTS {table_name}")

            print("Datenbank wurde erfolgreich geleert.")

        except Exception as e:
            print(f"Fehler beim Leeren der Datenbank: {e}")
    def test(self):
        return self.con.execute("SHOW TABLES").fetchdf()
    def clear_specifi_database(self, table):
        try:
            self.con.execute(f"DROP TABLE IF EXISTS {table}")
            print(f"Datenbank {table} wurde erfolgreich geleert.")
        except Exception as e:
            print(f"Fehler beim Leeren der Datenbank: {e}")

In [47]:
DataHub().clear_specifi_database("bronze_gmd")
DataHub().insert_gmd(gmd_df)
DataHub().preview_data()


DuckDB verbunden: c:\Diversification\data\bronze.db
Datenbank bronze_gmd wurde erfolgreich geleert.
DuckDB verbunden: c:\Diversification\data\bronze.db
DuckDB verbunden: c:\Diversification\data\bronze.db


,countryname,iso3,year,item_description,value,ingested_at
0,Aruba,ABW,1960,nGDP,NaN,2026-04-10 08:51:37.206467
1,Aruba,ABW,1961,nGDP,NaN,2026-04-10 08:51:37.206467
2,Aruba,ABW,1962,nGDP,NaN,2026-04-10 08:51:37.206467
3,Aruba,ABW,1963,nGDP,NaN,2026-04-10 08:51:37.206467
4,Aruba,ABW,1964,nGDP,NaN,2026-04-10 08:51:37.206467
...,...,...,...,...,...,...
4549035,Zimbabwe,ZWE,2026,income_group_code,2.0,2026-04-10 08:51:37.206467
4549036,Zimbabwe,ZWE,2027,income_group_code,2.0,2026-04-10 08:51:37.206467
4549037,Zimbabwe,ZWE,2028,income_group_code,2.0,2026-04-10 08:51:37.206467
4549038,Zimbabwe,ZWE,2029,income_group_code,2.0,2026-04-10 08:51:37.206467


In [3]:
testo = DataHub().preview_data()
testo[testo["year"] > 2000]

IOException: IO Error: Cannot open file "c:\diversification\data\bronze.db": Der Prozess kann nicht auf die Datei zugreifen, da sie von einem anderen Prozess verwendet wird.

File is already open in 
C:\Users\Konra\miniforge3\envs\gnn-data\python.exe (PID 3684)